# Long Phase 1: GPU Dialogue Generation

This notebook runs the separate exploratory long-dialogue protocol. It preserves the original Phase 1 factorial design: 24 stimuli x competition/cooperation/neutral x 3 generation seeds = 216 dialogues. Each dialogue stops only after at least 10,000 generated English-style words.

This is **not** the preregistered ten-turn MVP Phase 1. It uses a rolling context: when the rendered chat prompt exceeds 6,000 tokens, oldest complete messages are discarded. Results must be labelled exploratory.

Before running: Colab menu `Runtime` -> `Change runtime type` -> select `T4 GPU` or a stronger GPU. Google Drive is used for durable records so a stopped Colab session can resume safely.

## Put the code in Colab

Recommended GitHub workflow:

1. In GitHub Desktop, open the local `LANC` repository.
2. Commit all current changes, including `configs/phase1_long.yaml`, `src/agent_language_geometry/dialogue.py`, `src/agent_language_geometry/cli.py`, and this notebook.
3. Push that commit to GitHub.
4. Set `REPO_URL` below to your repository URL.

Alternative without GitHub: upload a zip of the full `LANC` folder to Colab, unzip it to `/content/LANC`, and skip the clone cell. Do not upload `.venv`, `results`, or `artifacts` folders.

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/shavrinaanastasia/LANC.git'
REPO = Path('/content/LANC')

if not REPO.exists():
    !git clone $REPO_URL $REPO
else:
    !git -C $REPO pull --ff-only

%cd $REPO
print('Repository:', REPO.resolve())

In [ ]:
# Do not install the repository's pinned CPU/GPU torch wheel over Colab's CUDA torch.
import sys
!{sys.executable} -m pip install -q PyYAML==6.0.1 scipy==1.13.1 scikit-learn==1.5.1 pandas==2.2.2 transformers==4.43.3
!{sys.executable} -m pip install -q -e . --no-deps

import torch
assert torch.cuda.is_available(), 'No GPU detected. Change the Colab runtime type to GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)

## Persist results on Google Drive

The full run can outlast a Colab session. After each completed dialogue, the code creates a separate immutable record in Drive. On resume, record hashes, the configuration hash, and the requested scope are checked before completed dialogues are skipped.

Keep `RUN_ID` unchanged after the first run. A changed run ID creates a different experiment; a changed configuration prevents resume.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

RUN_ID = 'phase1-long-10000w-v1'  # Choose once. Never change this when resuming.
DRIVE_RESULTS = Path('/content/drive/MyDrive/LANC-results')
DRIVE_RESULTS.mkdir(parents=True, exist_ok=True)
print('Results root:', DRIVE_RESULTS)

In [ ]:
# This derived config is deterministic. Recreate it identically before every resume.
import yaml

colab_config = {
    'extends': 'phase1_long.yaml',
    'run_id': RUN_ID,
    'paths': {'results_root': str(DRIVE_RESULTS)},
}
config_path = REPO / 'configs' / 'phase1_long_colab.yaml'
config_path.write_text(yaml.safe_dump(colab_config, sort_keys=False), encoding='utf-8')
print(config_path.read_text())
!python -m agent_language_geometry.cli validate-config --config configs/phase1_long_colab.yaml

In [ ]:
# Inspect progress before launching or resuming.
import json

run_dir = DRIVE_RESULTS / RUN_ID
records_dir = run_dir / 'dialogue_records'
records = sorted(records_dir.glob('*.json')) if records_dir.exists() else []
print(f'Completed immutable dialogue records: {len(records)} / 216')
if records:
    latest = json.loads(records[-1].read_text())
    print('Example saved record:', latest['interaction_frame'], latest['stimulus_id'], latest['generation_seed'])
    print('Words:', latest['protocol_metadata']['generated_words'])

## First launch

Run this cell once. It continues until all 216 dialogues are complete or the Colab session stops. Each completed dialogue prints a `saved completed/216` line and is already durable on Drive. Do not run this first-launch cell again after an interruption; use the resume cell instead.

In [ ]:
RUN_FIRST_LAUNCH = False  # Change to True once, then run this cell.
if RUN_FIRST_LAUNCH:
    !python -m agent_language_geometry.cli run-phase1-long --config configs/phase1_long_colab.yaml
else:
    print('Set RUN_FIRST_LAUNCH = True to begin the new GPU run.')

## Resume after a Colab restart

Reconnect Drive, rerun the repository, install, Drive, config, and progress cells above. Then run the next cell. It verifies completed records rather than regenerating them.

In [ ]:
RUN_RESUME = False  # Change to True only after an interrupted initial run.
if RUN_RESUME:
    !python -m agent_language_geometry.cli run-phase1-long --config configs/phase1_long_colab.yaml --resume
else:
    print('Set RUN_RESUME = True only to continue an existing GPU run.')

## Completion check and files

After all 216 records exist, the command creates one immutable `dialogues.jsonl`. That is the input for the later Schweinhart corpus-level analysis. Preserve both `dialogues.jsonl` and `dialogue_records/`; the latter proves what was completed before any resume.

In [ ]:
final_dialogues = run_dir / 'dialogues.jsonl'
print('Final JSONL exists:', final_dialogues.exists())
if final_dialogues.exists():
    print('Final output:', final_dialogues)
    print('Bytes:', final_dialogues.stat().st_size)